In [ ]:
#@title Install OmniVoice
import sys
%cd /content/

# Clean old files
!rm -rf omnivoice-colab OmniVoice

# Clone YOUR repo
!git clone https://github.com/hahyt6644-gif/omnivoice-colab.git
%cd /content/omnivoice-colab

# Clone OFFICIAL OmniVoice repo only
!git clone https://github.com/k2-fsa/OmniVoice.git

# Install ffmpeg
!apt-get update -qq && apt-get install -y ffmpeg

# Install requirements
!pip install -q -r colab.txt

# Extra packages needed for API
!pip install -q fastapi "uvicorn[standard]" python-multipart httpx requests aiofiles nest-asyncio python-dotenv pydub soundfile librosa

# Download latest app.py from YOUR repo
!wget -q -O app.py https://raw.githubusercontent.com/hahyt6644-gif/omnivoice-colab/main/app.py

# GPU check
import torch
from IPython.display import clear_output
clear_output()

print("========== SYSTEM INFO ==========")
if torch.cuda.is_available():
    print("✅ GPU ENABLED:", torch.cuda.get_device_name(0))
    print("✅ OmniVoice Installed Successfully")
else:
    print("\n" + "="*50)
    print("❌ CRITICAL ERROR: GPU NOT ENABLED!")
    print("="*50)
    print("OmniVoice requires a GPU to run. Please fix this by:")
    print("1. Clicking 'Runtime' in the top menu.")
    print("2. Clicking 'Change runtime type'.")
    print("3. Selecting 'T4 GPU' (or similar) under Hardware accelerator.")
    print("4. Clicking 'Save' and running this cell again.")
    print("="*50 + "\n")
    sys.exit("Execution stopped: GPU required to continue.")

In [ ]:
#@title Run OmniVoice APP + Cloudflare URL
%cd /content/omnivoice-colab

# Kill previous server
!pkill -9 -f "python app.py" || true
!pkill -9 -f "cloudflared" || true
!pkill -9 -f "uvicorn" || true

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1

import subprocess, time, re, requests, os, threading
from IPython.display import clear_output

print("🚀 Starting OmniVoice...")
# Run server in background and save logs to catch crashes
server = subprocess.Popen(['python', 'app.py'], stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)

# Wait for server
server_ready = False
for i in range(30):
    try:
        # Routing through 127.0.0.1 fixes Colab DNS drops
        requests.get('http://127.0.0.1:7860', timeout=2)
        server_ready = True
        break
    except:
        time.sleep(2)

if not server_ready:
    print("❌ Server failed to start. Printing crash logs:\n")
    os.system("tail -n 30 server.log")
else:
    print("\n🌍 Starting Cloudflare Tunnel...\n")
    tunnel = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )

    url = None
    start = time.time()
    while time.time() - start < 60:
        line = tunnel.stdout.readline()
        if not line: continue
        match = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            break

    clear_output()

    if url:
        print("=" * 60)
        print("✅ OMNIVOICE RUNNING")
        print("=" * 60)
        print(f"\n🌍 Public URL:\n{url}")
        print(f"\n🖥 UI:\n{url}/ui")
        print(f"\n🎙 Clone API:\n{url}/api/clone")
        print(f"\n🎨 Design API:\n{url}/api/design")
        print("\n⚠️ Keep notebook running. Restart cell = New URL")
        print("=" * 60)

        # Anti-freeze background thread
        def consume_tunnel_output(tunnel_proc):
            for _ in tunnel_proc.stdout: pass
        threading.Thread(target=consume_tunnel_output, args=(tunnel,), daemon=True).start()
    else:
        print("❌ Failed to get Cloudflare URL")